In [7]:
# Quiet TensorFlow logs and stabilize threading
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')  # 0=all,1=info,2=warning,3=error
try:
    import tensorflow as tf
    tf.config.threading.set_intra_op_parallelism_threads(0)
    tf.config.threading.set_inter_op_parallelism_threads(0)
except Exception as e:
    print('TF config setup skipped:', e)


# Kaggle-Ready MLP for COPD Risk

- Auto-detects `train.csv` and `test.csv` under `/kaggle/input/**` or local paths.
- No internet installs; uses Kaggle's preinstalled packages.
- Outputs `/kaggle/working/submission.csv`.


In [8]:
# Setup
import os, sys, random, json
import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

CV_SPLITS = 5
EPOCHS = 60
BATCH_SIZE = 256
PATIENCE = 8

print("TensorFlow:", tf.__version__)


TensorFlow: 2.18.0


In [9]:
# Locate data files robustly
from typing import Optional

def find_data_file(filename: str) -> Optional[Path]:
    candidates = []
    # Current workspace roots
    for p in [Path("."), Path(".."), Path("/kaggle/input")]:
        if p.exists():
            candidates.append(p)
    # Direct children that match
    for root in candidates:
        path = root / filename
        if path.exists():
            return path.resolve()
    # Walk under /kaggle/input for arbitrary dataset names
    ki = Path("/kaggle/input")
    if ki.exists():
        for dirpath, dirnames, filenames in os.walk(ki):
            if filename in filenames:
                return Path(dirpath) / filename
    # As last resort search shallowly in current dirs
    for root in candidates:
        for dirpath, dirnames, filenames in os.walk(root):
            if filename in filenames:
                return Path(dirpath) / filename
    return None

train_path = find_data_file("train.csv")
test_path  = find_data_file("test.csv")
print("train.csv:", train_path)
print("test.csv:", test_path)
assert train_path is not None and test_path is not None, "Could not find train.csv/test.csv. Please add your dataset to the notebook and re-run."

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)
print("Train shape:", df_train.shape, "Test shape:", df_test.shape)


train.csv: /kaggle/input/chronic-obstructive-pulmonary-disease-copd-risk/train.csv
test.csv: /kaggle/input/chronic-obstructive-pulmonary-disease-copd-risk/test.csv
Train shape: (44553, 27) Test shape: (11139, 26)


In [10]:
# Preprocessing with feature engineering

import numpy as np
import pandas as pd

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Basic derived measures
    h_cm = df['height_cm'].astype(float)
    w_kg = df['weight_kg'].astype(float)
    waist = df['waist_circumference_cm'].astype(float)
    sbp = df['bp_systolic'].astype(float)
    dbp = df['bp_diastolic'].astype(float)
    h_m = h_cm / 100.0

    # Avoid divide-by-zero
    h2 = h_m.pow(2)
    h2 = h2.replace(0, np.nan)

    df['bmi'] = w_kg / h2
    df['whr'] = waist / h_cm  # waist-to-height ratio
    df['pulse_pressure'] = sbp - dbp
    df['sbp_dbp_ratio'] = np.where(dbp == 0, np.nan, sbp / dbp)

    # Lipids
    total_chol = df['total_cholesterol'].astype(float)
    hdl = df['hdl_cholesterol'].astype(float)
    ldl = df['ldl_cholesterol'].astype(float)
    trig = df['triglycerides'].astype(float)
    df['chol_hdl_ratio'] = np.where(hdl == 0, np.nan, total_chol / hdl)
    df['non_hdl'] = total_chol - hdl
    df['ldl_hdl_ratio'] = np.where(hdl == 0, np.nan, ldl / hdl)

    # Enzymes and glucose relationships
    ast = df['ast_enzyme_level'].astype(float)
    alt = df['alt_enzyme_level'].astype(float)
    ggt = df['ggt_enzyme_level'].astype(float)
    glu = df['fasting_glucose'].astype(float)
    df['ast_alt_ratio'] = np.where(alt == 0, np.nan, ast / alt)
    df['enzymes_sum'] = ast + alt + ggt
    df['glucose_hdl_ratio'] = np.where(hdl == 0, np.nan, glu / hdl)

    # Log transforms for skewed lab values
    for col in ['triglycerides', 'ggt_enzyme_level', 'alt_enzyme_level', 'ast_enzyme_level', 'serum_creatinine']:
        if col in df.columns:
            df[f'log1p_{col}'] = np.log1p(df[col].astype(float).clip(lower=0))

    return df


def encode_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Map binary categoricals
    df['sex'] = df['sex'].map({'M': 1, 'F': 0}).astype('float32')
    df['oral_health_status'] = df['oral_health_status'].map({'Y': 1, 'N': 0}).astype('float32')
    df['tartar_presence'] = df['tartar_presence'].map({'Y': 1, 'N': 0}).astype('float32')
    return df

FEATURE_EXCLUDE = ['patient_id', 'has_copd_risk']
TARGET_COL = 'has_copd_risk'

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

preprocess_template = lambda: Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])


def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    df_fe = add_features(df)
    df_enc = encode_categoricals(df_fe)
    X = df_enc.drop(columns=[c for c in FEATURE_EXCLUDE if c in df_enc.columns], errors='ignore')
    return X


def split_features_labels(df: pd.DataFrame):
    X = build_feature_matrix(df)
    y = df[TARGET_COL].astype(int).values
    return X, y

X_df, y = split_features_labels(df_train)
print('Num features:', X_df.shape[1])


Num features: 40


In [11]:
# Threshold tuning utility (placed before CV to avoid NameError)
import numpy as np
from sklearn.metrics import f1_score

def tune_threshold(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.linspace(0.1, 0.9, 33)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_true, (y_prob >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

# Model (BatchNorm + L2 regularization)
from tensorflow.keras import regularizers

def build_mlp(input_dim: int) -> keras.Model:
    l2 = regularizers.l2(1e-4)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation='relu', kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu', kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu', kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ], name='mlp_binary')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=3e-4),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc'), 'accuracy']
    )
    return model


In [12]:
# Cross-Validation (F1) with per-fold pipeline and threshold tuning
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)

oof_prob = np.zeros_like(y, dtype=float)
fold_thresholds = []
fold_scores = []

# Prepare raw test frame features (same engineering as train, no target)
X_test_raw = build_feature_matrix(df_test)

test_fold_probs = []

for i, (tr, va) in enumerate(skf.split(X_df.values, y), start=1):
    X_tr_df, X_va_df = X_df.iloc[tr], X_df.iloc[va]
    y_tr, y_va = y[tr], y[va]

    # Per-fold preprocessing
    preprocess = preprocess_template()
    X_tr = preprocess.fit_transform(X_tr_df.values)
    X_va = preprocess.transform(X_va_df.values)
    X_test_fold = preprocess.transform(X_test_raw.values)

    # Class weights
    pos = (y_tr == 1).sum()
    neg = (y_tr == 0).sum()
    total = len(y_tr)
    class_weight = {0: total / (2.0 * neg), 1: total / (2.0 * pos)}

    model = build_mlp(X_tr.shape[1])

    es = keras.callbacks.EarlyStopping(monitor='val_auc', patience=PATIENCE, mode='max', restore_best_weights=True)
    rl = keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=max(2, PATIENCE//2), mode='max', min_lr=1e-5, verbose=0)

    model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[es, rl],
        class_weight=class_weight,
        verbose=0,
    )

    p_va = model.predict(X_va, batch_size=1024, verbose=0).ravel()
    t_opt, f1_opt = tune_threshold(y_va, p_va)
    yhat_va = (p_va >= t_opt).astype(int)

    oof_prob[va] = p_va
    fold_thresholds.append(t_opt)
    fold_scores.append(f1_opt)

    # Predict test with this fold preprocessor + model
    p_test_fold = model.predict(X_test_fold, batch_size=1024, verbose=0).ravel()
    test_fold_probs.append(p_test_fold)

    print(f"Fold {i}: F1={f1_opt:.4f} @ thr={t_opt:.3f}")

print("Per-fold F1:", np.round(fold_scores, 4))
print("Mean F1:", float(np.mean(fold_scores)))

# Global threshold tuned on OOF
thr_global, f1_global = tune_threshold(y, oof_prob)
print(f"OOF-tuned threshold={thr_global:.3f}, OOF F1={f1_global:.4f}")

# Average test probabilities across folds
test_pred_mean = np.mean(np.vstack(test_fold_probs), axis=0)


Fold 1: F1=0.7199 @ thr=0.400
Fold 2: F1=0.7199 @ thr=0.450
Fold 3: F1=0.7185 @ thr=0.500
Fold 4: F1=0.7236 @ thr=0.525
Fold 5: F1=0.7196 @ thr=0.450
Per-fold F1: [0.7199 0.7199 0.7185 0.7236 0.7196]
Mean F1: 0.7202934324869986
OOF-tuned threshold=0.475, OOF F1=0.7195


In [13]:
# Create submission from fold ensemble predictions
from pathlib import Path

TEST_ID_COL = 'patient_id'

# Use OOF-tuned global threshold for final binarization
final_thr = thr_global if 'thr_global' in globals() else 0.5
print(f"Final threshold: {final_thr:.3f}")

# If not available (e.g., CV not run), fallback to simple pipeline on full data
if 'test_pred_mean' not in globals():
    print('CV predictions not found; falling back to full-data training...')
    preprocess_full = preprocess_template()
    X_full = preprocess_full.fit_transform(X_df.values)
    model_full = build_mlp(X_full.shape[1])
    es = keras.callbacks.EarlyStopping(monitor='val_auc', patience=PATIENCE, mode='max', restore_best_weights=True)
    rl = keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=max(2, PATIENCE//2), mode='max', min_lr=1e-5, verbose=0)
    X_tr, X_va, y_tr, y_va = train_test_split(X_full, y, test_size=0.1, stratify=y, random_state=SEED)
    model_full.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es, rl], verbose=0)
    X_test_raw = build_feature_matrix(df_test)
    X_test = preprocess_full.transform(X_test_raw.values)
    test_pred_mean = model_full.predict(X_test, batch_size=1024, verbose=0).ravel()

# Build submission
sub = pd.DataFrame({
    'patient_id': df_test[TEST_ID_COL],
    'has_copd_risk': (test_pred_mean >= final_thr).astype(int)
})
out_path = Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else Path('submission.csv')
sub.to_csv(out_path, index=False)
print('Wrote:', out_path, '\n', sub.head())


Final threshold: 0.475
Wrote: /kaggle/working/submission.csv 
    patient_id  has_copd_risk
0       42427              0
1       27412              0
2       19283              1
3       45261              1
4       11155              1
